# Notebook 02 — Paper Figures

**Purpose:** Generate publication-quality figures for the MDPI paper.
Each figure cell produces one plot saved to `data/figures/`.

**Prerequisites (all scripts must have been run):**
```
python src/analysis/scripts/01_load_validate.py
python src/analysis/scripts/02_compute_stats.py
python src/analysis/scripts/03_statistical_tests.py
```

**Inputs:**
- `data/processed/stats_paper_s{1,2,3}.parquet`
- `data/processed/statistical_tests.json`

**Outputs:** `data/figures/*.pdf` (vector) and `*.png` (raster)

**NTP caveats:**
- S1/S2 `latency_ms` is NTP-inflated (two clocks). Values are valid for
  *relative* protocol comparison only. Mention NTP offset δ in the paper.
- S3 uses `cin_create_ms` (NTP-free, same machine). This is the primary latency
  metric. Do **not** plot `latency_ms` for S3.

**Authors:** João Parreira, Pedro Barbeiro

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

REPO_ROOT = Path().resolve().parents[2]
DATA_PROCESSED = REPO_ROOT / 'data' / 'processed'
FIGURES_DIR = REPO_ROOT / 'data' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Protocol display names and colours (consistent across all figures)
PROTO_COLORS = {
    'mqtt':      '#2196F3',
    'websocket': '#4CAF50',
    'http':      '#FF9800',
    'coap':      '#9C27B0',
}
PROTO_LABELS = {
    'mqtt':      'MQTT',
    'websocket': 'WebSocket',
    'http':      'HTTP',
    'coap':      'CoAP',
}

# Publication style
plt.rcParams.update({
    'font.family':    'serif',
    'font.size':      10,
    'axes.titlesize': 10,
    'axes.labelsize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.dpi':     150,
    'savefig.dpi':    300,
    'savefig.bbox':   'tight',
})
sns.set_theme(style='whitegrid')

def save_fig(name: str) -> None:
    """Save current figure as PDF and PNG to data/figures/."""
    for ext in ('pdf', 'png'):
        path = FIGURES_DIR / f'{name}.{ext}'
        plt.savefig(path)
    print(f'Saved: {name}.pdf / .png')

print(f'Figures will be saved to: {FIGURES_DIR}')

In [ ]:
# Load per-run stats
s1 = pd.read_parquet(DATA_PROCESSED / 'stats_paper_s1.parquet')
s2 = pd.read_parquet(DATA_PROCESSED / 'stats_paper_s2.parquet')
s3 = pd.read_parquet(DATA_PROCESSED / 'stats_paper_s3.parquet')

# Load statistical test results
with open(DATA_PROCESSED / 'statistical_tests.json') as f:
    stat_tests = json.load(f)

protocols = sorted(set(s1['protocol'].unique()) | set(s2['protocol'].unique()) | set(s3['protocol'].unique()))
print(f'Protocols: {protocols}')
print(f'S1 runs: {len(s1)}, S2 runs: {len(s2)}, S3 runs: {len(s3)}')

## Figure 1 — S1/S2 Mean latency by protocol (box plot)

NTP-inflated values are shown. Mention in caption that absolute values include
the NTP offset δ ≈ mean(latency_ms_S3_raw) but relative comparison is valid.

In [ ]:
combined = pd.concat([
    s1.assign(scenario='S1 (4 msg/s)'),
    s2.assign(scenario='S2 (16 msg/s)'),
])

fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(
    data=combined,
    x='scenario', y='lat_mean', hue='protocol',
    palette=PROTO_COLORS,
    ax=ax, width=0.6, linewidth=0.8,
)
ax.set_xlabel('')
ax.set_ylabel('Mean latency per run (ms)')
ax.set_title('S1/S2 — End-to-end latency by protocol (NTP-inflated)')
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, [PROTO_LABELS.get(l, l) for l in labels], title='Protocol')
plt.tight_layout()
save_fig('fig1_s1s2_latency_boxplot')
plt.show()

## Figure 2 — S3 cin_create_ms by protocol (box plot)

This is the primary latency figure for S3 (command round-trip, NTP-free).

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
sns.boxplot(
    data=s3,
    x='protocol', y='cin_mean',
    palette=PROTO_COLORS,
    ax=ax, width=0.5, linewidth=0.8,
    order=protocols,
)
ax.set_xticklabels([PROTO_LABELS.get(p, p) for p in protocols])
ax.set_xlabel('')
ax.set_ylabel('Mean CIN round-trip per run (ms)')
ax.set_title('S3 — Command latency (Streamlit -> CSE, NTP-free)')
plt.tight_layout()
save_fig('fig2_s3_cin_latency_boxplot')
plt.show()

## Figure 3 — Packet loss by protocol and scenario

In [ ]:
all_stats = pd.concat([
    s1.assign(scenario='S1'),
    s2.assign(scenario='S2'),
    s3.assign(scenario='S3'),
])

summary_loss = (
    all_stats.groupby(['protocol', 'scenario'])['packet_loss_frac']
    .agg(['mean', 'std'])
    .reset_index()
)

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(
    data=summary_loss, x='scenario', y='mean', hue='protocol',
    palette=PROTO_COLORS, ax=ax,
    errorbar=None,
)
ax.set_ylabel('Mean packet loss (fraction)')
ax.set_xlabel('')
ax.set_title('Packet loss by protocol and scenario')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, [PROTO_LABELS.get(l, l) for l in labels], title='Protocol')
plt.tight_layout()
save_fig('fig3_packet_loss')
plt.show()

## Figure 4 — Protocol overhead (%) by scenario

In [ ]:
summary_oh = (
    all_stats.groupby(['protocol', 'scenario'])['overhead_pct_mean']
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(
    data=summary_oh, x='scenario', y='overhead_pct_mean', hue='protocol',
    palette=PROTO_COLORS, ax=ax, errorbar=None,
)
ax.set_ylabel('Protocol overhead (%)')
ax.set_xlabel('')
ax.set_title('Protocol overhead by scenario\n(header / (header + payload) x 100)')
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, [PROTO_LABELS.get(l, l) for l in labels], title='Protocol')
plt.tight_layout()
save_fig('fig4_overhead')
plt.show()

## Figure 5 — S1 Throughput by protocol

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
sns.boxplot(
    data=s1, x='protocol', y='tput_msg_per_s',
    palette=PROTO_COLORS, ax=ax, width=0.5, linewidth=0.8,
    order=protocols,
)
ax.set_xticklabels([PROTO_LABELS.get(p, p) for p in protocols])
ax.set_xlabel('')
ax.set_ylabel('Delivered messages / s')
ax.set_title('S1 (4 msg/s) — Throughput by protocol')
ax.axhline(4, color='red', linestyle='--', linewidth=1, label='target (4 msg/s)')
ax.legend()
plt.tight_layout()
save_fig('fig5_s1_throughput')
plt.show()

## Figure 6 — S3 Jitter (cin_std) by protocol

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
sns.boxplot(
    data=s3, x='protocol', y='cin_std',
    palette=PROTO_COLORS, ax=ax, width=0.5, linewidth=0.8,
    order=protocols,
)
ax.set_xticklabels([PROTO_LABELS.get(p, p) for p in protocols])
ax.set_xlabel('')
ax.set_ylabel('Jitter (std dev of cin_create_ms, ms)')
ax.set_title('S3 — Command latency jitter by protocol')
plt.tight_layout()
save_fig('fig6_s3_jitter')
plt.show()

## Table 1 — Summary statistics for paper

In [ ]:
def fmt_mean_std(mean, std, decimals=1):
    if pd.isna(mean):
        return 'N/A'
    if pd.isna(std):
        return f'{mean:.{decimals}f}'
    return f'{mean:.{decimals}f} +/- {std:.{decimals}f}'

rows = []
for proto in protocols:
    # S1
    s1p = s1[s1['protocol'] == proto]
    s1_lat = fmt_mean_std(s1p['lat_mean'].mean(), s1p['lat_mean'].std())
    s1_loss = f"{s1p['packet_loss_frac'].mean()*100:.1f}%"
    s1_tput = fmt_mean_std(s1p['tput_msg_per_s'].mean(), s1p['tput_msg_per_s'].std())
    s1_oh = fmt_mean_std(s1p['overhead_pct_mean'].mean(), None)
    # S3
    s3p = s3[s3['protocol'] == proto]
    s3_cin = fmt_mean_std(s3p['cin_mean'].mean(), s3p['cin_mean'].std())
    s3_loss = f"{s3p['packet_loss_frac'].mean()*100:.1f}%"

    rows.append({
        'Protocol': PROTO_LABELS.get(proto, proto),
        'S1 lat (ms)*': s1_lat,
        'S1 loss': s1_loss,
        'S1 tput (msg/s)': s1_tput,
        'S1 overhead': s1_oh,
        'S3 CIN RTT (ms)': s3_cin,
        'S3 loss': s3_loss,
    })

table = pd.DataFrame(rows).set_index('Protocol')
print('Table 1 — Summary statistics')
print('* S1/S2 latency is NTP-inflated; compare relative values only')
print()
print(table.to_string())

## Statistical test results summary

In [ ]:
for result in stat_tests:
    group = result.get('group', '?')
    metric = result.get('metric', '?')
    if 'error' in result:
        print(f"[SKIP] {group:4s} {metric:25s}: {result['error']}")
    else:
        kw = result['kruskal_wallis']
        sig = 'SIG' if kw['significant'] else '   '
        print(f"[{sig}] {group:4s} {metric:25s}: H={kw['H']:.2f}  p={kw['p']:.4f}")